# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [2]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [3]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [4]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|███████████| 5/5 [01:02<00:00, 12.54s/it]


In [5]:
len(deals)

50

In [6]:
deals[44].describe()

'Title: 2" Queen Memory Foam Mattress Topper for $33 + free shipping\nDetails: Apply coupon code "AEUS08" to save an extra $8. Buy Now at AliExpress\nFeatures: \nURL: https://www.dealnews.com/2-Queen-Memory-Foam-Mattress-Topper-for-33-free-shipping/21754635.html?iref=rss-c196'

In [7]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [8]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [9]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Pioneer PN40-551-24U 40" 1080p LED HD Smart TV for $100 + free shipping
Details: Grab this deal at its best price. Buy Now at eBay
Features: 1080p resolution Xumo TV 3 HDMI inputs Model: PN40-551-24U
URL: https://www.dealnews.com/products/Pioneer/Pioneer-PN40-551-24-U-40-1080-p-LED-HD-Smart-TV/486841.html?iref=rss-c142

Title: Best Buy Back To School Deals: Up to 50

In [10]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [11]:
result = get_recommendations()

In [12]:
len(result.deals)

5

In [13]:
result.deals[1]

Deal(product_description="The Vibeat IDW26 is a stylish fitness smartwatch that boasts a vibrant 1.83-inch TFT-LCD display. Designed for active individuals, it features a comfortable silicone sport band and is IP68 waterproof, ensuring durability during workouts. The watch is equipped with Alexa voice control and supports over 100 exercise modes, allowing you to track various activities effortlessly. With its combination of functionality and modern design, it's an ideal companion for those looking to enhance their fitness journey.", price=35.0, url='https://www.dealnews.com/Vibeat-IDW26-1-83-Fitness-Smartwatch-with-Alarm-for-35-free-shipping/21754617.html?iref=rss-c142')

In [14]:
from agents.scanner_agent import ScannerAgent

In [15]:
agent = ScannerAgent()
result = agent.scan()

In [16]:
result

DealSelection(deals=[Deal(product_description='The Pioneer PN40-551-24U is a vibrant 40" LED HD Smart TV with 1080p resolution that offers a fantastic viewing experience. Equipped with three HDMI inputs, this Xumo TV allows for extensive connectivity options for your media devices. This model combines quality with convenience, making it an excellent choice for both entertainment and smart home integration.', price=100.0, url='https://www.dealnews.com/products/Pioneer/Pioneer-PN40-551-24-U-40-1080-p-LED-HD-Smart-TV/486841.html?iref=rss-c142'), Deal(product_description='The Refurbished Unlocked Samsung Galaxy S22 is a powerful smartphone featuring a Qualcomm SM8450 Snapdragon 8 Gen 1 8-core CPU and a stunning 6.1" 2340x1080 AMOLED touchscreen display. With a remarkable 108MP back camera and a 40MP front camera, this device excels at capturing high-quality images. Equipped with Android 12, an 8GB RAM, and 128GB of storage, it offers robust performance and versatility, all backed by a 1-ye